# Qwen-Image-2.1 models Dataset builder

This notebook downloads only the three Qwen-Image-2.1 INT8 files. It does not download ComfyUI, cloudflared, or create archives.

The only top-level output folder is:

`qwen-image-21-int8-models/`

In [ ]:
from __future__ import annotations

import json
import shutil
import time
from pathlib import Path

from huggingface_hub import hf_hub_download

WORKING = Path('/kaggle/working')
DATASET_ROOT = WORKING / 'qwen-image-21-int8-models'
RUNTIME_ROOT = Path('/tmp/qwen-models-dataset')
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_REPO = 'Comfy-Org/Qwen-Image-2.1'
MODEL_FILES = {
    'diffusion_models': 'diffusion_models/qwen_image_2.1_int8_convrot.safetensors',
    'text_encoders': 'text_encoders/qwen3vl_8b_int8_convrot.safetensors',
    'vae': 'vae/qwen_image_2.1_vae_bf16.safetensors',
}
MINIMUM_SIZES = {
    'diffusion_models': 7_000_000_000,
    'text_encoders': 9_000_000_000,
    'vae': 600_000_000,
}

started = time.monotonic()
print('DATASET_ROOT', DATASET_ROOT, flush=True)
print('MODEL_REPO', MODEL_REPO, flush=True)
for folder, filename in MODEL_FILES.items():
    print('EXPECTED', folder, filename, flush=True)


In [ ]:
manifest = {
    'model_repo': MODEL_REPO,
    'files': {},
    'root_only': True,
    'comfyui_included': False,
}
sizes = {}
for folder, filename in MODEL_FILES.items():
    target = DATASET_ROOT / folder / Path(filename).name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists() or target.stat().st_size < MINIMUM_SIZES[folder]:
        print('DOWNLOADING', filename, flush=True)
        downloaded = Path(hf_hub_download(
            repo_id=MODEL_REPO,
            filename=filename,
            local_dir=str(DATASET_ROOT),
        ))
        if downloaded != target:
            shutil.copy2(downloaded, target)
    if target.stat().st_size < MINIMUM_SIZES[folder]:
        raise RuntimeError(f'File is too small: {target} ({target.stat().st_size} bytes)')
    sizes[filename] = target.stat().st_size
    manifest['files'][filename] = {
        'relative_path': str(target.relative_to(DATASET_ROOT)),
        'size_bytes': target.stat().st_size,
    }
    print('READY', filename, target.stat().st_size, flush=True)

# Keep the output clean: no Hugging Face cache or partial metadata.
shutil.rmtree(DATASET_ROOT / '.cache', ignore_errors=True)
manifest['elapsed_seconds'] = round(time.monotonic() - started, 2)
(DATASET_ROOT / 'dataset_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n')
print('MANIFEST', json.dumps(manifest, ensure_ascii=False), flush=True)


In [ ]:
expected = {str(DATASET_ROOT / folder / Path(filename).name) for folder, filename in MODEL_FILES.items()}
actual = {str(path) for path in DATASET_ROOT.rglob('*') if path.is_file()}
missing = sorted(expected - actual)
if missing:
    raise RuntimeError(f'Missing model files: {missing}')

# Ensure no ComfyUI checkout or second top-level output directory was created.
top_level = sorted(path.name for path in WORKING.iterdir())
allowed = {'qwen-image-21-int8-models', '__notebook__.ipynb'}
unexpected = [name for name in top_level if name not in allowed]
if unexpected or 'qwen-image-21-int8-models' not in top_level:
    raise RuntimeError(f'Unexpected top-level output: {top_level}')

print('TOP_LEVEL_OUTPUT', top_level, flush=True)
print('KAGGLE_SERVICE_FILES', ['__notebook__.ipynb'], flush=True)
print('TOTAL_BYTES', sum(path.stat().st_size for path in DATASET_ROOT.rglob('*') if path.is_file()), flush=True)
print('QWEN_MODELS_DATASET_READY', DATASET_ROOT, flush=True)
